# Leapfrog algorithm theory

We are using this in a NVE ensemble (not NVT). For this ensemble the code given by prof. Keller uses different steps to construct the Verlet algorithm. The snippet of the code is given in the cell below:

If we take a look at the code snippets of the steps we can see how the equations fit into the leapfrog integrator.
1. __A-step__: in this step the positions are updated for a full ($\Delta t$) or half time step ($\frac{\Delta t}{2}$), using velocities at current ($t$) time. This code should be used to calculate the position update using velocities that are half a step ahead ($v(t+\frac{\Delta}{2})$).
2. __B-step__: in this step the velocities are updated for a for a full ($\Delta t$) or half time step ($\frac{\Delta t}{2}$) using the force that was calculated at time $t$. 

The convenient thing about both A and B steps that they already have an option to switch on a half a time step using `half_step == True`. Now we just have to construct the code together so that the leapfrog algorithm is used properly.

In [ ]:
def A_step(ps: ParticleSystem, sim: SimulationParameters, half_step=False):
    """
    Performs the A-step (position update) of an MD integration scheme.

    This step updates particle positions using the current velocities:
    r(t + Δt) = r(t) + v(t) * Δt

    Parameters:
        - ps (ParticleSystem): The particle system containing positions and velocities.
        - sim (SimulationParameters): Simulation settings, including the time step.
        - half_step (bool): If True, performs a half step (Δt / 2) instead of a full step.

    Returns:
        None. Updates ps.position in-place.
    """
    # set time step, depending on whether a half- or full step is performed
    if half_step == True:
        dt = 0.5 * sim.dt
    else:
        dt = sim.dt
        
    ps.position = ps.position + ps.velocity * dt
    
    return None    

def B_step(ps: ParticleSystem, sim: SimulationParameters, half_step=False):
    """
    Performs the B-step (velocity update) of an MD integration scheme.

    This step updates particle velocities using the current forces:
    v(t + Δt) = v(t) + 1/m * Δt * F(t) 
 
    Parameters:
        - ps (ParticleSystem): The particle system containing positions and velocities.
        - sim (SimulationParameters): Simulation settings, including the time step.
        - half_step (bool): If True, performs a half step (Δt / 2) instead of a full step.

    Returns:
        None. Updates ps.velocity in-place.
    """
    # set time step, depending on whether a half- or full step is performed
    if half_step == True:
        dt = 0.5 * sim.dt
    else:
        dt = sim.dt
        
    # (1/ps.mass)[:, np.newaxis] = explicit reshaping to avoid
    # broadcasting issues when multiplying (N,) with (N,3) elementwise
    # now it is explicit: (N,1) * (N,3)
    ps.velocity = ps.velocity + (1/ps.mass)[:, np.newaxis]* dt * ps.force 
    
    return None   

Here is the snippet of the BAB Verlet integrator code:

In [ ]:
def simulate_NVE_step(ps: ParticleSystem, sim: SimulationParameters):
    """
    Performs a single time step of molecular dynamics in the NVE ensemble
    using the velocity Verlet integrator in BAB form (half-step B, full-step A, half-step B).

    The steps are:
    1. Half-step velocity update (B-step)
    2. Full-step position update (A-step)
    3. Force recalculation based on new positions
    4. Second half-step velocity update (B-step)
    5. Apply periodic boundary conditions

    This corresponds to a time-symmetric, second-order accurate integrator for Newtonian dynamics.

    Parameters:
        - ps (ParticleSystem): The particle system containing positions, velocities, and forces.
        - sim (SimulationParameters): Simulation parameters including time step.

    Returns:
        None. Updates ps.position, ps.velocity, and ps.force in-place.
    """
    B_step(ps, sim, half_step=True)   # update velocity by a half-step
    A_step(ps, sim, half_step=False)  # update position by a full time step
    calculate_force(ps, sim)          # udpate force  
    B_step(ps, sim, half_step=True)   # update velocity by a second half-step

    apply_periodic_boundary(ps, sim)
        
    return None  

## Leapfrog algorithm outline:
The idea of the leapfrog algorithm is to evaluate the positions and velocities at different points in time, they are offset by half a timestep and thus they leapfrog over each other. Initially we need to offset the velocity by half a time-step (just in the initialization part). See below for the explanation of the algorithm.

These are the equations for the leapfrog algorithm:
$$
\mathbf{v}\left(t + \frac{1}{2}\Delta t\right)
=
\mathbf{v}\left(t - \frac{1}{2}\Delta t\right)
+
\frac{\mathbf{F}(t)}{m}\,\Delta t
$$

$$
\mathbf{r}(t + \Delta t)
=
\mathbf{r}(t)
+
\mathbf{v}\left(t + \frac{1}{2}\Delta t\right)\,\Delta t
$$

Points 1 and 2 are done before the loop.
1. Calculate the force according to the initial positions F(0). This is already done in the run script in the LJ_gas_run_MD.py (line 120): 
```python
# calculate force according to initial positions
calculate_force(ps, sim)
```
2. Do a half step velocity update (B-step) using `half_step == True`. This updates $v(0)$-->v($0+\frac{1}{2}\Delta t$)

__In the actual MD loop__:

3. Do a full step position update (A-step, `half_step == False`) using the half-step velocity update v($0+\frac{1}{2}\Delta t$). This moves positions from $r(0)$ --> $r(0+\Delta t)$
4. Calculate the force at time $F(0+\Delta t$) from the new positions in step 3.
5. Do a full B step: since we have the force we can move the velocities to a full time-step ahead: $v(0+\frac{1}{2}\Delta t)$-->$v(0+\frac{3}{2}\Delta t)$
6. Because we have moved the particles and updated the velocities we need to apply periodic boundary conditions.
So we see that we have positions at integer times but the velocities at half-integer times, so they leapfrog cleanly over each other.
7. To track the kinetic energy at the integer time (at which we store positions) we need to average the velocities before and after the B step. That way energy calculations are synchronized (hasn't been done yet).

__Averaging the velocities to get accurate energy results__

In the leapfrog integrator the velocities are calculated at half-time steps and not integer time step values, so if we just leave it like this the kinetic energy calculation would be shifted since they wouldn't be calculated at the same point in time as the positions. To correct this we need to (linearly) interpolate between the velocity values for each time step. This can be done using this equation:
$$
\mathbf{v}(t + \Delta t)
\approx
\frac{
\mathbf{v}\left(t + \frac{1}{2}\Delta t\right)
+
\mathbf{v}\left(t + \frac{3}{2}\Delta t\right)
}{2}
$$ 

In the code this is implemnted in such a way:
1. Before the leapfrog step is calculated the program copies the velocity (which is at half-integer step, $t+\frac{1}{2} \Delta t$). This is saved as `v_before`.
2. Simulate a leapfrog step. In the step we obtain the velocity at $t+\frac{3}{2} \Delta t$. This is saved as `v_after`.
3. Now we calculate the average value between the two (as per equation above). So we get a velocity estimate at integer time step, `v_sync`.
4. We for the next step we also the save the velocity that will be used in the next iteration, `v_actual`.
5. Crucially, the program temporarily swaps the `v_actual` with `v_sync`. This is executed with the line `ps.velocity = v_sync`.
6. The code calls the functions `potential_energy`, `kinetic_energy`, `instantaneous_temperature` and `ideal_gas_pressure` to calculate the energies in the current integration step, using `v_sync`.
7. The synced velocity (at integer time step, `v_sync`) is again swapped with the actual velocity that it's used for integration, `v_actual`.

# Comparison between the different integrators, random seed for the same initial positions
To properly compare the Velocity Verlet and Leap frog integrator (in the NVE system), and also between the NVE and NVT ensembles, we need to set a random seed so that the simulation boxes initialize at the same exact positions every time.

If we take a look at the code snippet of the function `initialize_positions`:
```python
def initialize_positions(ps: ParticleSystem, box_length_in_nm: float):
    """Initialize particle positions uniformly in a cubic box."""
    ps.position[:] = np.random.uniform(0, box_length_in_nm, size=(ps.n, 3))
```
This function uses the random module to draw numbers uniformly and to place them in the box. Using `np.random.seed(integer)` this would always generate the same numbers, therefore generates the same positions. This seed can be added in the _run_ script, before we call the `initialize_positions` function. Analogous should be done also for `initialize_velocities` function. 

# Notes what can we plot (some pointers given by the tutor)
The project assigments:
* Implement a leapfrog integrator in the MD code

* Compare position, velocity, and energy evolution to the BAOAB scheme: _we have to read on the differences between the NVT and NVE ensemble. Collisions of the particles in the heat baths in NVT can introduce random noise. This can be shown in a way that we change the coupling constant tau of the thermostat._ 

* Analyze differences in energy conservation and trajectory stability: _we can start with 2 simulation boxes, both of them are initialized at the exact same initial positions. Then we run them with different ensembles/different integrators (but parameters should be the same). Then we can do a RMSE (root mean square error):_
$$ \text{RMSE}(t) = \sqrt{\frac{1}{N} \sum_{i=1}^{N} \left| \mathbf{r}_i(t) - \mathbf{r}_i^{\text{ref}}(t) \right|^2} $$
$N$: Total number of particles.
$\mathbf{r}_i(t)$: The $(x, y, z)$ position of particle $i$ in your test simulation.
$\mathbf{r}_i^{\text{ref}}(t)$: The $(x, y, z)$ position of particle $i$ in a "reference" simulation.
 
__This next part was generated by AI__: RMSE quantifies this drift by calculating the average physical distance between the particles in your simulation and a "reference" simulation at any given time $t$. Mathematically, the basic Verlet and Leapfrog algorithms generate the exact same trajectory for positions.If you start them with the exact same initial conditions and the same $\Delta t$, their position RMSE compared to each other will be essentially zero (differing only by microscopic floating-point computer rounding errors). So the more physically meaningful comparisons are __NVT vs NVE__ (the thermostat injects randomness, so trajectories will diverge), __different time steps__ (larger dt means more numerical error accumulating over time), __short vs long runs__ (does energy drift appear only at long timescales?). _Also it's good to visualize the particle trajectories in VMD._


 

* Study long-term energy drift for different MD setups and integrators: _creating stable and non-stable MD setups with leapfrog and verlet. Which parameters make it stable and which unstable?_

* Analyze the performance for different time steps: here we will probably see that at a certain time step the integrator will become unstable. We can figure out at what that time step is and why.

## RMSE analysis, important changes:

In the run script the following code was added:<br>
`np.save(file_name_base + "_pos.npy", position_trajectory)`.<br>
This saves the positions in a binary `.npy` file, because in the original script we only had `.npy` files for the energies.<br>

Position trajectories are stored as (from initialization line):<br> `position_trajectory = np.zeros((sim.n_steps+1, n_particles, 3))`.<br>
This creates a matrix with three axes of different dimensions: `(t,N,xyz)` meaning `(timestep, particle index, xyz positions)`.<br>
Therefore axis = 0 is time (size n_steps+1), axis = 1 is particle index (from 0->N-1, size N), axis = 2 are particle positions (x,y,z; size 3).

Calculating the differences between the particle positions is relatively easy with vectorization. Looking at the RMSE equation the order of operations are:
- the difference between the particle positions and reference particle positions for every particle:<br>`difference = traj_file - traj_file_ref`, shape of `(n_steps+1, n_particles, 3)`
- square it and sum of the differences across all xyz positions (along axis 2):<br>`diff_sum_xyz = np.sum(difference**2, axis = 2)`, shape of `(n_steps+1, n_particles)`
- averaging it over all particles, meaning calculating the sum along along all particles (axis 1) and diving by number of particles, and at last taking the square root:<br>
`diff_sum_particles = np.sum(diff_sum_xyz, axis = 1) # sum over particles, shape (n_steps+1,)`
`rmse = np.sqrt(diff_sum_particles/n_particles_)`

$$ \text{RMSE}(t) = \sqrt{\frac{1}{N} \sum_{i=1}^{N} \left| \mathbf{r}_i(t) - \mathbf{r}_i^{\text{ref}}(t) \right|^2} $$

I have also added code to save the parameters from the run script to the analysis script:
```python
params = {
    "n_particles": n_particles,
    "mass_argon": mass_argon,
    "sigma_argon": sigma_argon,
    "epsilon_argon": epsilon_argon,
    "dt":dt,
    "n_steps": n_steps,
    "temperature": temperature,
    "box_length": box_length,
    "tau_thermostat": tau_thermostat,
    "random_seed": random_seed
    }
np.save(file_name_base + "_params.npy", params) # save parameters as .npy file
```
<br>In the analysis script we load it using this line:<br>
`params = np.load("my_simulation_NVE_vVerlet_params.npy", allow_pickle=True).item()`

What does `allow_pickle = True` do? The binary `.npy` files can only store numerical arrays. Pickling converts an non-numerical object into a binary format which can be saved in the file. Later it can be unpickled to reconstruct the original Python object. Pickle is by default disabled in the `np.load` function since it can execute arbitrary (potentially malicious) code. Here the source us trusted, so we allow pickle to recontruct the original parameters dictionary. Calling `.item()` extracts the the dictionary (otherwise it return a 0-dimensional object array). This way we recover the original dictionary.

__PROBLEM: Spikes in the RMSE analysis caused by periodic boundary conditions__

<img src="NVE_Leapfrog_NVE_vVerlet_RMSE_PBC_artefacts.png" width="500"><br>

When a particle crosses the box boundary, its position wraps from ~50 nm back to ~0 nm (or vice versa). If in one simulation a particle is at 49.9 nm and in the other it's at 0.1 nm, physically they're only 0.2 nm apart — but the RMSE calculation sees a raw difference of 49.8 nm, creating a huge spike.
The fix is to apply the minimum image convention to the differences, exactly the same way it's done in calculate_force in LJ_gas.py:
```python
L = params_analysis["box_length"]

difference = traj_file - traj_file_ref          # shape (n_steps+1, N, 3)
difference -= L * np.rint(difference / L)       # wrap differences to [-L/2, L/2]
```
This means if the apparent distance is more than half the box length, the particle must have crossed the boundary, so we need to measure from the other side instead. The same logic is used for example in the original `calculate_force` and `potential_energy` function.

The result looks like this:<br>
<img src="NVE_Leapfrog_NVE_vVerlet_RMSE_PBC_solved.png" width="500"><br>


__Optional__:<br>
To compare RMSE effectively between a reference run (e.g. vVerlet with dt = 0.01 ps) and comparison run (e.g. Leapfrog with dt = 0.1 ps) we need two things:
- same total simulation time in ps (e.g. 100 ps)
- same number of time steps (frames) at the same time points

According to AI we have to subsample the finer trajectory: for example if we run 10000 steps with dt = 0.01 as reference and then run 1000 steps of dt = 0.1 then we subsample at every 10th frame from the reference run. This way we get the same total simulation time of 100 ps and same number of time steps (frames) for the RMSE calculation.<br>
I think maybe we can skip this as coding this again changes a bit the `LJ_gas_run_MD.py` script, namely changing n_steps from a constant varibale to something like: `n_steps = int(total_time / dt)`<br>
__This is actally quite handy if we want to compare the time-steps, since we need to have the same total simulation time (e.g. 100ps) (last bullet point)__